# Preparing data for Model 2

### keep adding new seasons and updating the leagues and df

In [85]:
import pandas as pd
import requests
from io import StringIO

# Add new seasons here as they become available
SEASONS = ['1920', '2021', '2122', '2223', '2324', '2425']

# League codes from football-data.co.uk
LEAGUES = {
    'premier_league': 'E0',
    'bundesliga':     'D1',
    'la_liga':        'SP1',
    'serie_a':        'I1'
}

BASE_URL = "https://www.football-data.co.uk/mmz4281/{season}/{league}.csv"

In [86]:
def download_season(league_code, season):
    url = BASE_URL.format(season=season, league=league_code)
    response = requests.get(url)
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text), encoding='latin-1')
        df['season'] = season
        return df
    else:
        print(f"Failed: {url}")
        return None

all_dfs = []
for league_name, league_code in LEAGUES.items():
    for season in SEASONS:
        df = download_season(league_code, season)
        if df is not None:
            df['league'] = league_name
            all_dfs.append(df)

raw = pd.concat(all_dfs, ignore_index=True)
print(raw.shape)

(8676, 135)


In [87]:
COLS = ['Date', 'HomeTeam', 'AwayTeam', 'HS', 'AS', 'MaxH', 'MaxD', 'MaxA', 'league', 'season']

raw_final = raw[COLS]
raw_final.columns = ['date', 'home_team', 'away_team', 'home_shots', 'away_shots',
               'odds_h', 'odds_d', 'odds_a', 'league', 'season']

# Convert the 'date' column to datetime objects
raw_final['date'] = pd.to_datetime(raw_final['date'], format='%d/%m/%Y', errors='coerce')

/tmp/ipykernel_1312/824191156.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw_final['date'] = pd.to_datetime(raw_final['date'], format='%d/%m/%Y', errors='coerce')


### change all irregularities here

In [88]:
date_to_drop = pd.to_datetime('14/12/2024', format='%d/%m/%Y')
home_team_to_drop = 'Union Berlin'
away_team_to_drop = 'Bochum'

# Create a boolean mask to identify the row(s) to drop
mask_to_drop = ((raw_final['date'] == date_to_drop) &
                (raw_final['home_team'] == home_team_to_drop) &
                (raw_final['away_team'] == away_team_to_drop))

# Check if the row to drop exists before dropping
if mask_to_drop.any():
    raw_final = raw_final[~mask_to_drop].copy() # Use .copy() to avoid SettingWithCopyWarning
    print(f"Dropped row: {home_team_to_drop} vs {away_team_to_drop} on {date_to_drop.strftime('%d/%m/%Y')}")
else:
    print(f"Row to drop (date: {date_to_drop.strftime('%d/%m/%Y')}, home_team: {home_team_to_drop}, away_team: {away_team_to_drop}) not found.")

date_to_update = pd.to_datetime('10/01/2022', format='%d/%m/%Y')
home_team_to_update = 'Torino'
away_team_to_update = 'Fiorentina'

# Create a boolean mask to identify the row(s) to update
mask_to_update = ((raw_final['date'] == date_to_update) &
                  (raw_final['home_team'] == home_team_to_update) &
                  (raw_final['away_team'] == away_team_to_update))

# Check if the row to update exists before updating
if mask_to_update.any():
    raw_final.loc[mask_to_update, 'odds_h'] = 3.0
    raw_final.loc[mask_to_update, 'odds_d'] = 3.3
    raw_final.loc[mask_to_update, 'odds_a'] = 2.4

    print("Odds updated successfully. Here's the updated row:")
    display(raw_final.loc[mask_to_update])
else:
    print(f"Row to update (date: {date_to_update.strftime('%d/%m/%Y')}, home_team: {home_team_to_update}, away_team: {away_team_to_update}) not found.")

Dropped row: Union Berlin vs Bochum on 14/12/2024
Odds updated successfully. Here's the updated row:


,date,home_team,away_team,home_shots,away_shots,odds_h,odds_d,odds_a,league,season
7359,2022-01-10,Torino,Fiorentina,9.0,10.0,3.0,3.3,2.4,serie_a,2122


In [89]:
# change odds to probabilities

raw_final['prob_h'] = (1 / raw_final['odds_h'])
raw_final['prob_d'] = (1 / raw_final['odds_d'])
raw_final['prob_a'] = (1 / raw_final['odds_a'])

# Normalise to remove bookmaker margin
total = raw_final['prob_h'] + raw_final['prob_d'] + raw_final['prob_a']
raw_final['prob_h'] = raw_final['prob_h'] / total
raw_final['prob_d'] = raw_final['prob_d'] / total
raw_final['prob_a'] = raw_final['prob_a'] / total

### Save it as a df onto the colab folder here

In [90]:
from google.colab import drive
drive.mount('/content/drive')

raw_final.to_csv('/content/drive/MyDrive/shots_odds/Model_2/football_data.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Make columns

In [91]:
data = raw_final

In [92]:
# 1

home_df = data[['date', 'season', 'league', 'home_team', 'away_team', 'home_shots', 'away_shots']].copy()
home_df = home_df.rename(columns={
    'home_team': 'team',
    'away_team': 'opponent',
    'home_shots': 'shots_scored',
    'away_shots': 'shots_conceded'
})
home_df['venue'] = 'home'

away_df = data[['date', 'season', 'league', 'away_team', 'home_team', 'away_shots', 'home_shots']].copy()
away_df = away_df.rename(columns={
    'away_team': 'team',
    'home_team': 'opponent',
    'away_shots': 'shots_scored',
    'home_shots': 'shots_conceded'
})
away_df['venue'] = 'away'

long_df = pd.concat([home_df, away_df], ignore_index=True)
long_df = long_df.sort_values(['team', 'season', 'date']).reset_index(drop=True)

print(long_df.shape)
print(long_df.head())

(17350, 8)
        date season   league    team    opponent  shots_scored  \
0 2019-08-18   1920  la_liga  Alaves     Levante           9.0   
1 2019-08-25   1920  la_liga  Alaves     Espanol          10.0   
2 2019-08-31   1920  la_liga  Alaves      Getafe           5.0   
3 2019-09-15   1920  la_liga  Alaves     Sevilla           7.0   
4 2019-09-22   1920  la_liga  Alaves  Ath Bilbao           5.0   

   shots_conceded venue  
0            16.0  home  
1             5.0  home  
2            15.0  away  
3            19.0  home  
4            13.0  away  


In [93]:
# 2

# For each league-season, compute expanding average shots conceded up to each date
# We use all teams' shots conceded in that league-season up to that point

league_season_expanding = (data.sort_values('date')
                           .groupby(['league', 'season']))

def expanding_league_avg(group):
    # Total shots in each game for that league/season up to that point
    group = group.copy()
    group['expanding_avg_home_shots'] = group['home_shots'].expanding().mean().shift(1)
    group['expanding_avg_away_shots'] = group['away_shots'].expanding().mean().shift(1)
    group['expanding_neutral'] = (group['expanding_avg_home_shots'] + group['expanding_avg_away_shots']) / 2
    group['expanding_avg_shots_conc'] = (group['home_shots'] + group['away_shots']).expanding().mean().shift(1) / 2
    return group

league_season_stats = (data.sort_values('date')
                       .groupby(['league', 'season'], group_keys=False)
                       .apply(expanding_league_avg))

league_season_stats = league_season_stats[['date', 'league', 'season',
                                           'expanding_avg_home_shots',
                                           'expanding_avg_away_shots',
                                           'expanding_neutral',
                                           'expanding_avg_shots_conc']]

print(league_season_stats.head(10))

        date          league season  expanding_avg_home_shots  \
0 2019-08-09  premier_league   1920                       NaN   
1 2019-08-10  premier_league   1920                 15.000000   
2 2019-08-10  premier_league   1920                 10.000000   
3 2019-08-10  premier_league   1920                 11.000000   
4 2019-08-10  premier_league   1920                 10.750000   
5 2019-08-10  premier_league   1920                  9.800000   
6 2019-08-10  premier_league   1920                 10.000000   
7 2019-08-11  premier_league   1920                 13.000000   
8 2019-08-11  premier_league   1920                 13.250000   
9 2019-08-11  premier_league   1920                 12.777778   

   expanding_avg_away_shots  expanding_neutral  expanding_avg_shots_conc  
0                       NaN                NaN                       NaN  
1                 12.000000          13.500000                 13.500000  
2                 13.000000          11.500000             

/tmp/ipykernel_1312/3564970184.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(expanding_league_avg))


In [94]:
# 3

# For each team, compute rolling avg shots conceded over last 5 games, same season only, min 3 games

def rolling_shots_conc(group):
    group = group.copy()
    group['opp_rolling_shots_conc'] = (group['shots_conceded']
                                        .shift(1)
                                        .rolling(window=5, min_periods=3)
                                        .mean())
    return group

long_df = (long_df.sort_values(['team', 'season', 'date'])
           .groupby(['team', 'season'], group_keys=False)
           .apply(rolling_shots_conc))

print(long_df[['team', 'date', 'season', 'shots_conceded', 'opp_rolling_shots_conc']].head(20))

      team       date season  shots_conceded  opp_rolling_shots_conc
0   Alaves 2019-08-18   1920            16.0                     NaN
1   Alaves 2019-08-25   1920             5.0                     NaN
2   Alaves 2019-08-31   1920            15.0                     NaN
3   Alaves 2019-09-15   1920            19.0                   12.00
4   Alaves 2019-09-22   1920            13.0                   13.75
5   Alaves 2019-09-26   1920            21.0                   13.60
6   Alaves 2019-09-29   1920             5.0                   14.60
7   Alaves 2019-10-05   1920            11.0                   14.60
8   Alaves 2019-10-20   1920            14.0                   13.80
9   Alaves 2019-10-25   1920            19.0                   12.80
10  Alaves 2019-10-29   1920             9.0                   14.00
11  Alaves 2019-11-03   1920            19.0                   11.60
12  Alaves 2019-11-09   1920             7.0                   14.40
13  Alaves 2019-11-24   1920      

/tmp/ipykernel_1312/583097776.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(rolling_shots_conc))


In [95]:
# 4

# Take the last expanding value for each date/league/season
# (after all games on that date have been processed)
league_season_stats_dedup = (league_season_stats
                             .sort_values('date')
                             .groupby(['date', 'league', 'season'])
                             .last()
                             .reset_index())

# Now merge
long_df = long_df.merge(
    league_season_stats_dedup,
    on=['date', 'league', 'season'],
    how='left'
)

print(long_df.shape)
print(long_df.head())

(17350, 13)
        date season   league    team    opponent  shots_scored  \
0 2019-08-18   1920  la_liga  Alaves     Levante           9.0   
1 2019-08-25   1920  la_liga  Alaves     Espanol          10.0   
2 2019-08-31   1920  la_liga  Alaves      Getafe           5.0   
3 2019-09-15   1920  la_liga  Alaves     Sevilla           7.0   
4 2019-09-22   1920  la_liga  Alaves  Ath Bilbao           5.0   

   shots_conceded venue  opp_rolling_shots_conc  expanding_avg_home_shots  \
0            16.0  home                     NaN                 12.166667   
1             5.0  home                     NaN                 11.250000   
2            15.0  away                     NaN                 12.375000   
3            19.0  home                   12.00                 12.351351   
4            13.0  away                   13.75                 12.400000   

   expanding_avg_away_shots  expanding_neutral  expanding_avg_shots_conc  
0                 11.500000          11.833333       

In [96]:
# 5

long_df['neutral_shots'] = long_df.apply(
    lambda row: row['shots_scored'] - (row['expanding_avg_home_shots'] - row['expanding_neutral'])
    if row['venue'] == 'home'
    else row['shots_scored'] + (row['expanding_neutral'] - row['expanding_avg_away_shots']),
    axis=1
)

print(long_df[['team', 'date', 'venue', 'shots_scored', 'neutral_shots']].head(20))

      team       date venue  shots_scored  neutral_shots
0   Alaves 2019-08-18  home           9.0       8.666667
1   Alaves 2019-08-25  home          10.0      10.156250
2   Alaves 2019-08-31  away           5.0       6.270833
3   Alaves 2019-09-15  home           7.0       6.121622
4   Alaves 2019-09-22  away           5.0       5.933333
5   Alaves 2019-09-26  away           5.0       5.903509
6   Alaves 2019-09-29  home          11.0      10.046154
7   Alaves 2019-10-05  away          10.0      11.077465
8   Alaves 2019-10-20  home          12.0      11.064706
9   Alaves 2019-10-25  away          12.0      13.038889
10  Alaves 2019-10-29  home          11.0       9.915000
11  Alaves 2019-11-03  away          11.0      12.101770
12  Alaves 2019-11-09  home          14.0      12.929167
13  Alaves 2019-11-24  away           7.0       8.145522
14  Alaves 2019-11-30  home           8.0       6.839161
15  Alaves 2019-12-07  away           5.0       6.160000
16  Alaves 2019-12-13  home    

In [97]:
# 6

# We need the opponent's rolling shots conceded at the time of each match
# This is the opp_rolling_shots_conc from the opponent's perspective

opp_stats = long_df[['team', 'date', 'season', 'opp_rolling_shots_conc']].copy()
opp_stats = opp_stats.rename(columns={
    'team': 'opponent',
    'opp_rolling_shots_conc': 'opp_def_strength'
})

long_df = long_df.merge(
    opp_stats,
    on=['opponent', 'date', 'season'],
    how='left'
)

print(long_df[['team', 'opponent', 'date', 'opp_def_strength']].head(20))

      team     opponent       date  opp_def_strength
0   Alaves      Levante 2019-08-18               NaN
1   Alaves      Espanol 2019-08-25               NaN
2   Alaves       Getafe 2019-08-31               NaN
3   Alaves      Sevilla 2019-09-15              7.00
4   Alaves   Ath Bilbao 2019-09-22              7.25
5   Alaves     Sociedad 2019-09-26             14.00
6   Alaves     Mallorca 2019-09-29             11.80
7   Alaves     Valencia 2019-10-05             14.80
8   Alaves        Celta 2019-10-20             11.20
9   Alaves   Villarreal 2019-10-25             12.00
10  Alaves   Ath Madrid 2019-10-29              9.60
11  Alaves      Osasuna 2019-11-03              9.00
12  Alaves   Valladolid 2019-11-09             14.60
13  Alaves        Eibar 2019-11-24              9.60
14  Alaves  Real Madrid 2019-11-30              9.60
15  Alaves      Granada 2019-12-07             12.00
16  Alaves      Leganes 2019-12-13             10.80
17  Alaves    Barcelona 2019-12-21            

In [98]:
# 7

# Apply opponent strength scaling
long_df['adj_shots'] = (long_df['neutral_shots'] *
                        (long_df['expanding_avg_shots_conc'] / long_df['opp_def_strength']))

# Now compute rolling mean of adj_shots over last 5 games, same season, min 3 games
def rolling_adj_shots(group):
    group = group.copy()
    group['rolling_adj_shots'] = (group['adj_shots']
                                   .shift(1)
                                   .rolling(window=5, min_periods=3)
                                   .mean())
    return group

long_df = (long_df.sort_values(['team', 'season', 'date'])
           .groupby(['team', 'season'], group_keys=False)
           .apply(rolling_adj_shots))

print(long_df[['team', 'date', 'season', 'adj_shots', 'rolling_adj_shots']].head(20))

      team       date season  adj_shots  rolling_adj_shots
0   Alaves 2019-08-18   1920        NaN                NaN
1   Alaves 2019-08-25   1920        NaN                NaN
2   Alaves 2019-08-31   1920        NaN                NaN
3   Alaves 2019-09-15   1920  10.033314                NaN
4   Alaves 2019-09-22   1920   9.384215                NaN
5   Alaves 2019-09-26   1920   4.775332                NaN
6   Alaves 2019-09-29   1920   9.666312           8.064287
7   Alaves 2019-10-05   1920   8.438819           8.464793
8   Alaves 2019-10-20   1920  11.047272           8.459598
9   Alaves 2019-10-25   1920  12.344689           8.662390
10  Alaves 2019-10-29   1920  11.727586           9.254485
11  Alaves 2019-11-03   1920  15.427674          10.644936
12  Alaves 2019-11-09   1920  10.099067          11.797208
13  Alaves 2019-11-24   1920   9.786151          12.129257
14  Alaves 2019-11-30   1920   8.185272          11.877033
15  Alaves 2019-12-07   1920   5.964933          11.0451

/tmp/ipykernel_1312/2410282560.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(rolling_adj_shots))


In [99]:
# 8

# Split long_df back into home and away
home_features = long_df[long_df['venue'] == 'home'][['team', 'date', 'season',
                                                       'rolling_adj_shots',
                                                       'opp_rolling_shots_conc']].copy()
home_features.columns = ['home_team', 'date', 'season',
                          'home_adj_shots',
                          'home_adj_shots_conc']

away_features = long_df[long_df['venue'] == 'away'][['team', 'date', 'season',
                                                       'rolling_adj_shots',
                                                       'opp_rolling_shots_conc']].copy()
away_features.columns = ['away_team', 'date', 'season',
                          'away_adj_shots',
                          'away_adj_shots_conc']

# Merge onto original dataframe
data = data.merge(home_features, on=['date', 'home_team', 'season'], how='left')
data = data.merge(away_features, on=['date', 'away_team', 'season'], how='left')

print(data.shape)
print(data[['date', 'home_team', 'away_team',
            'home_adj_shots', 'home_adj_shots_conc',
            'away_adj_shots', 'away_adj_shots_conc']].head(20))

(8675, 17)
         date         home_team         away_team  home_adj_shots  \
0  2019-08-09         Liverpool           Norwich             NaN   
1  2019-08-10          West Ham          Man City             NaN   
2  2019-08-10       Bournemouth  Sheffield United             NaN   
3  2019-08-10           Burnley       Southampton             NaN   
4  2019-08-10    Crystal Palace           Everton             NaN   
5  2019-08-10           Watford          Brighton             NaN   
6  2019-08-10         Tottenham       Aston Villa             NaN   
7  2019-08-11         Leicester            Wolves             NaN   
8  2019-08-11         Newcastle           Arsenal             NaN   
9  2019-08-11        Man United           Chelsea             NaN   
10 2019-08-17           Arsenal           Burnley             NaN   
11 2019-08-17       Aston Villa       Bournemouth             NaN   
12 2019-08-17          Brighton          West Ham             NaN   
13 2019-08-17          

In [100]:
# 9

data = data.dropna(subset=['home_adj_shots', 'home_adj_shots_conc',
                            'away_adj_shots', 'away_adj_shots_conc'])

print(f"Final shape: {data.shape}")
print(data[['home_adj_shots', 'home_adj_shots_conc',
            'away_adj_shots', 'away_adj_shots_conc']].describe())

Final shape: (7258, 17)
       home_adj_shots  home_adj_shots_conc  away_adj_shots  \
count     7258.000000          7258.000000     7258.000000   
mean        12.665074            12.375489       12.681958   
std          3.197667             3.010798        3.227740   
min          3.872601             3.600000        4.053458   
25%         10.400583            10.200000       10.402037   
50%         12.339645            12.200000       12.339515   
75%         14.616197            14.400000       14.622940   
max         27.774609            26.000000       28.471457   

       away_adj_shots_conc  
count          7258.000000  
mean             12.153706  
std               2.993204  
min               3.400000  
25%              10.000000  
50%              12.000000  
75%              14.000000  
max              27.400000  


In [101]:
data

,date,home_team,away_team,home_shots,away_shots,odds_h,odds_d,odds_a,league,season,prob_h,prob_d,prob_a,home_adj_shots,home_adj_shots_conc,away_adj_shots,away_adj_shots_conc
60,2019-09-28,Sheffield United,Liverpool,12.0,16.0,10.00,5.40,1.39,premier_league,1920,0.099541,0.184335,0.716123,10.732373,11.2,14.529708,10.2
61,2019-09-28,Aston Villa,Burnley,16.0,10.0,2.45,3.47,3.18,premier_league,1920,0.403797,0.285102,0.311101,9.055650,16.0,7.815182,14.6
62,2019-09-28,Bournemouth,West Ham,13.0,17.0,2.58,3.65,2.90,premier_league,1920,0.385133,0.272231,0.342636,12.930449,18.6,10.338506,13.2
63,2019-09-28,Chelsea,Brighton,24.0,8.0,1.47,4.80,8.50,premier_league,1920,0.676045,0.207039,0.116916,15.237339,8.6,14.105633,10.6
64,2019-09-28,Crystal Palace,Norwich,15.0,10.0,2.00,3.85,4.00,premier_league,1920,0.495177,0.257235,0.247588,11.312785,14.8,9.831641,17.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8670,2025-05-25,Empoli,Verona,14.0,5.0,2.04,3.40,4.35,serie_a,2425,0.483333,0.290000,0.226667,9.447004,8.6,9.002569,8.8
8671,2025-05-25,Lazio,Lecce,26.0,9.0,1.64,3.86,8.00,serie_a,2425,0.613546,0.260677,0.125777,14.938184,8.0,13.531812,11.2
8672,2025-05-25,Torino,Roma,11.0,15.0,6.33,4.33,1.60,serie_a,2425,0.155808,0.227775,0.616417,12.552135,9.4,13.491459,13.2
8673,2025-05-25,Udinese,Fiorentina,12.0,25.0,3.81,3.75,2.06,serie_a,2425,0.258698,0.262837,0.478465,14.850600,11.8,7.987065,9.4


In [102]:
from google.colab import drive
drive.mount('/content/drive')

data.to_csv('/content/drive/MyDrive/shots_odds/Model_2/football_data_adjcols.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
